# Scenario 4 — Explanation-Augmented (CoT) Fine-Tuning
## Notebook 2 of 2 — Decoder CoT FT (Qwen2.5-1.5B)  ← Main Experiment

**This is the centrepiece of Scenario 4.**
The paper's core finding (Yao et al. 2025, arXiv 2505.00034):

> Qwen2.5-1.5B **label-only FT: 38.8%** → **CoT FT: 86.0%** F1
> Teaching the model to reason first closes most of the gap with BERT (99%)
> AND produces human-readable explanations BERT can never give.

This notebook runs **both** Qwen configurations so the delta is visible in one place:
1. Qwen label-only FT  → baseline (no reasoning)
2. Qwen CoT FT         → main experiment (reasoning + label)

### Shared Conditions (identical to Notebook 1)
| Factor | Value |
|---|---|
| Dataset | `puyang2025/seven-phishing-email-datasets` — SpamAssassin subset |
| Split | 80 / 10 / 10 stratified, `random_state=42` |
| Metrics | Accuracy, Precision, Recall, F1, ms/sample |
| Max token length | 256 |
| Epochs | 3 |

### Fixes vs original
- Qwen label-only baseline added (was missing — original compared BERT vs Qwen CoT)
- Robust label extraction using `rfind` (original `in` check misfired on negations)
- Gradient accumulation (effective batch=8 vs original batch=2 with no accumulation)
- Max length 512 for CoT sequences (original 384 truncated rationales)
- Rationale caching to disk (no re-generation on re-runs)
- Sample explanation display after inference


In [ ]:
!nvidia-smi
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall -q
!pip install transformers accelerate bitsandbytes peft datasets scikit-learn pandas numpy tqdm -q


In [1]:
import os, re, time, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                              precision_score, recall_score, classification_report)
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          get_linear_schedule_with_warmup, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

# ── Shared constants (same as Notebook 1) ──
SEED    = 42
MAX_LEN = 256
EPOCHS  = 3
torch.manual_seed(SEED)

DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
DECODER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BNB_CFG      = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── Shared evaluation function (same as Notebook 1) ──
def evaluate(y_true, y_pred, name="", ms=None):
    result = {
        "Model"     : name,
        "Accuracy"  : f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision" : f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall"    : f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1"        : f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}",
    }
    if ms:
        result["ms/sample"] = f"{ms:.2f}"
    return result

# ── Robust label extractor (fix vs original) ──
def extract_label(decoded_text: str) -> int:
    """
    Find the LAST occurrence of PHISHING or LEGITIMATE.
    Handles negations like 'this is NOT phishing' correctly.
    Defaults to 0 (legitimate) if neither keyword found.
    """
    text       = decoded_text.upper()
    last_phish = text.rfind("PHISHING")
    last_legit = text.rfind("LEGITIMATE")
    if last_phish == -1 and last_legit == -1:
        return 0
    return 1 if last_phish > last_legit else 0


Device : cuda
GPU    : Tesla T4


In [3]:
# ── Shared dataset loading (same in both notebooks) ──
print("Loading puyang2025/seven-phishing-email-datasets...")
ds     = load_dataset("puyang2025/seven-phishing-email-datasets", split="train")
df_all = ds.to_pandas()

# Inspect actual columns
print("Columns:", df_all.columns.tolist())

# Build text from whatever columns exist
if "body" in df_all.columns:
    df_all["text"] = (df_all["subject"].fillna("") + " " + df_all["body"].fillna("")).str.strip()
elif "text" in df_all.columns:
    df_all["text"] = df_all["text"].fillna("").str.strip()
else:
    # fallback: concatenate all string columns except label/dataset_name
    str_cols = [c for c in df_all.columns if c not in ("label", "dataset_name")
                and df_all[c].dtype == object]
    print("Using string cols for text:", str_cols)
    df_all["text"] = df_all[str_cols].fillna("").agg(" ".join, axis=1).str.strip()

df_all = (df_all[["text", "label", "dataset_name"]]
          .drop_duplicates("text").dropna().reset_index(drop=True))
df_all["label"] = df_all["label"].astype(int)

print(f"Total rows : {len(df_all):,}")
print(df_all["dataset_name"].value_counts().to_string())

Loading puyang2025/seven-phishing-email-datasets...
Columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']
Total rows : 161,480
dataset_name
TREC-05     44259
TREC-07     42746
CEAS-08     30728
Enron       23799
TREC-06     13081
Assassin     4576
Ling         2291


In [4]:
# ── Shared split (same as Notebook 1) ──
# 80 / 10 / 10  stratified  random_state=42
df_assassin = df_all[df_all["dataset_name"] == "Assassin"].reset_index(drop=True)

df_tr,  df_tmp = train_test_split(df_assassin, test_size=0.20,
                                   stratify=df_assassin["label"], random_state=SEED)
df_val, df_te  = train_test_split(df_tmp,       test_size=0.50,
                                   stratify=df_tmp["label"],       random_state=SEED)

print(f"Train : {len(df_tr):,}  |  positive rate: {df_tr['label'].mean():.3f}")
print(f"Val   : {len(df_val):,}  |  positive rate: {df_val['label'].mean():.3f}")
print(f"Test  : {len(df_te):,}  |  positive rate: {df_te['label'].mean():.3f}")


Train : 3,660  |  positive rate: 0.293
Val   : 458  |  positive rate: 0.295
Test  : 458  |  positive rate: 0.293


In [5]:
# ══════════════════════════════════════════════════════════════
# RATIONALE GENERATOR  (rule-based — no API key required)
# Generates a 2-3 sentence explanation + classification label
# Used to build CoT training targets
# ══════════════════════════════════════════════════════════════

def generate_rationale(text: str, label: int) -> str:
    t       = text.lower()
    signals = []

    if any(w in t for w in ["click here", "verify", "confirm your", "log in", "sign in"]):
        signals.append("contains an urgency call-to-action typical of credential-harvesting")

    if any(w in t for w in ["account", "password", "suspend", "limited", "unusual activity"]):
        signals.append("references an account security threat used to induce panic")

    if re.search(r'https?://\S+', t):
        signals.append("includes a URL that may redirect to a spoofed login page")

    if any(w in t for w in ["dear customer", "dear user", "dear valued"]):
        signals.append("uses a generic salutation instead of the recipient name")

    if any(w in t for w in ["congratulations", "winner", "prize", "lottery", "million"]):
        signals.append("contains reward language characteristic of advance-fee fraud")

    if any(w in t for w in ["invoice", "payment", "overdue", "bank transfer", "wire"]):
        signals.append("references financial transactions used in business email compromise")

    if label == 1:
        if signals:
            body = ("This email shows multiple phishing indicators. "
                    "It " + "; it ".join(signals) + ". "
                    "These patterns are consistent with a social engineering attempt.")
        else:
            body = ("This email creates a sense of urgency to push the reader into immediate action. "
                    "The overall tone and structure are consistent with phishing campaigns "
                    "that rely on pressure rather than specific technical deception.")
        return body + " Classification: PHISHING"
    else:
        if signals:
            body = ("This email contains some phrases that could appear suspicious in isolation. "
                    "However, the overall context, tone, and structure suggest a legitimate message. "
                    "No deceptive intent is evident when the full content is considered.")
        else:
            body = ("This email uses clear professional language with no deceptive indicators. "
                    "The content, tone, and structure are all consistent with legitimate correspondence. "
                    "There are no urgency triggers, spoofed links, or suspicious requests.")
        return body + " Classification: LEGITIMATE"


# Quick test
sample_text  = "Dear customer, your account has been suspended. Click here to verify."
sample_label = 1
print("Sample rationale (phishing):")
print(generate_rationale(sample_text, sample_label))
print()
sample_text2  = "Hi John, please find attached the meeting notes from Tuesday."
sample_label2 = 0
print("Sample rationale (legitimate):")
print(generate_rationale(sample_text2, sample_label2))


Sample rationale (phishing):
This email shows multiple phishing indicators. It contains an urgency call-to-action typical of credential-harvesting; it references an account security threat used to induce panic; it uses a generic salutation instead of the recipient name. These patterns are consistent with a social engineering attempt. Classification: PHISHING

Sample rationale (legitimate):
This email uses clear professional language with no deceptive indicators. The content, tone, and structure are all consistent with legitimate correspondence. There are no urgency triggers, spoofed links, or suspicious requests. Classification: LEGITIMATE


In [6]:
# Generate rationales for training set and cache to disk
CACHE_PATH = "/kaggle/working/rationale_cache.csv"

if os.path.exists(CACHE_PATH):
    print(f"Loading cached rationales from {CACHE_PATH}")
    cache     = pd.read_csv(CACHE_PATH)
    df_tr     = df_tr.merge(cache[["text", "rationale"]], on="text", how="left")
    missing   = df_tr["rationale"].isna().sum()
    print(f"  Loaded {len(cache):,} | Missing: {missing:,}")
else:
    df_tr   = df_tr.copy()
    missing = len(df_tr)

if missing > 0:
    print(f"Generating {missing:,} rationales...")
    if "rationale" not in df_tr.columns:
        df_tr["rationale"] = df_tr.apply(
            lambda r: generate_rationale(r["text"], r["label"]), axis=1)
    else:
        mask = df_tr["rationale"].isna()
        df_tr.loc[mask, "rationale"] = df_tr[mask].apply(
            lambda r: generate_rationale(r["text"], r["label"]), axis=1)
    df_tr[["text", "rationale"]].to_csv(CACHE_PATH, index=False)
    print(f"  Saved to {CACHE_PATH}")

print(f"\nTotal training rationales ready: {len(df_tr):,}")
print("\nSample:")
s = df_tr.iloc[0]
print(f"  Label: {s['label']}")
print(f"  {s['rationale']}")


Generating 3,660 rationales...
  Saved to /kaggle/working/rationale_cache.csv

Total training rationales ready: 3,660

Sample:
  Label: 1
  This email creates a sense of urgency to push the reader into immediate action. The overall tone and structure are consistent with phishing campaigns that rely on pressure rather than specific technical deception. Classification: PHISHING


In [7]:
# ── Label-only dataset (Qwen baseline) ──
class LabelOnlyDataset(Dataset):
    """Trains Qwen to output a single word: PHISHING or LEGITIMATE."""
    LABEL_MAP = {1: "PHISHING", 0: "LEGITIMATE"}

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.samples   = list(zip(texts, labels))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        text, label = self.samples[i]
        prompt    = f"Classify this email as PHISHING or LEGITIMATE:\n{text[:240]}\n\nAnswer:"
        target    = f" {self.LABEL_MAP[label]}"
        full_text = prompt + target

        enc = self.tokenizer(
            full_text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt")

        lbl = enc["input_ids"].clone().squeeze()
        prompt_len = len(self.tokenizer(prompt, add_special_tokens=False)["input_ids"])
        lbl[:prompt_len] = -100   # supervise label token only

        return {k: v.squeeze() for k, v in enc.items()}, lbl


# ── CoT dataset (main experiment) ──
class CoTDataset(Dataset):
    """Trains Qwen to output a rationale then a classification label."""
    COT_MAX_LEN = 512   # rationales need more room than labels

    def __init__(self, texts, rationales, tokenizer):
        self.tokenizer = tokenizer
        self.samples   = list(zip(texts, rationales))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        text, rationale = self.samples[i]
        prompt    = f"Analyse this email for phishing indicators:\n{text[:360]}\n\nAnalysis:"
        full_text = prompt + " " + rationale

        enc = self.tokenizer(
            full_text, padding="max_length", truncation=True,
            max_length=self.COT_MAX_LEN, return_tensors="pt")

        lbl = enc["input_ids"].clone().squeeze()
        prompt_len = len(self.tokenizer(prompt, add_special_tokens=False)["input_ids"])
        lbl[:prompt_len] = -100   # supervise rationale+label only

        return {k: v.squeeze() for k, v in enc.items()}, lbl


In [8]:
# ══════════════════════════════════════════════════════════════
# PART A — Qwen Label-Only FT  (baseline — no reasoning)
# Paper target: ~38.8% F1
# ══════════════════════════════════════════════════════════════
print("Loading Qwen2.5-1.5B for label-only fine-tuning...")

tok = AutoTokenizer.from_pretrained(DECODER_MODEL, trust_remote_code=True, padding_side="right")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model_lo = AutoModelForCausalLM.from_pretrained(
    DECODER_MODEL, quantization_config=BNB_CFG,
    device_map="auto", trust_remote_code=True)

lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1,
                      bias="none", target_modules=["q_proj", "v_proj"])
model_lo = get_peft_model(model_lo, lora_cfg)
model_lo.print_trainable_parameters()

BATCH      = 4
GRAD_ACCUM = 4   # effective batch = 16
LR         = 2e-4

tr_dl_lo = DataLoader(
    LabelOnlyDataset(df_tr["text"].values, df_tr["label"].values, tok),
    batch_size=BATCH, shuffle=True)

opt_lo   = AdamW(filter(lambda p: p.requires_grad, model_lo.parameters()), lr=LR)
sched_lo = get_linear_schedule_with_warmup(
    opt_lo, (len(tr_dl_lo)//GRAD_ACCUM)//10, (len(tr_dl_lo)//GRAD_ACCUM)*EPOCHS)

for ep in range(1, EPOCHS + 1):
    model_lo.train(); total = 0; opt_lo.zero_grad()
    for step, (be, lbl) in enumerate(tqdm(tr_dl_lo, desc=f"LabelOnly ep{ep}"), 1):
        be  = {k: v.to(DEVICE) for k, v in be.items()}
        out = model_lo(**be, labels=lbl.to(DEVICE))
        (out.loss / GRAD_ACCUM).backward()
        total += out.loss.item()
        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model_lo.parameters(), 1.0)
            opt_lo.step(); sched_lo.step(); opt_lo.zero_grad()
    print(f"  Epoch {ep} loss: {total/len(tr_dl_lo):.4f}")


Loading Qwen2.5-1.5B for label-only fine-tuning...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Epoch 1 loss: 0.7142


LabelOnly ep2:   0%|          | 0/915 [00:00<?, ?it/s]

  Epoch 2 loss: 0.0006


LabelOnly ep3:   0%|          | 0/915 [00:00<?, ?it/s]

  Epoch 3 loss: 0.0004


In [9]:
PROMPT_LO = "Classify this email as PHISHING or LEGITIMATE:\n{}\n\nAnswer:"

model_lo.eval(); preds_lo = []
t0 = time.time()

for text in tqdm(df_te["text"].values, desc="LabelOnly inference"):
    inputs = tok(PROMPT_LO.format(text[:240]),
                 return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)
    with torch.no_grad():
        out = model_lo.generate(
            **inputs, max_new_tokens=5,
            pad_token_id=tok.eos_token_id, do_sample=False)
    decoded = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    preds_lo.append(extract_label(decoded))

ms_lo = (time.time() - t0) / len(df_te) * 1000
RESULT_QWEN_LO = evaluate(df_te["label"].values, preds_lo,
                           "Qwen2.5-1.5B (label-only FT)", ms_lo)

print("\n" + "="*60)
print("PART A RESULT — Qwen Label-Only FT")
print("="*60)
print(pd.DataFrame([RESULT_QWEN_LO]).to_string(index=False))
print("\n→ Paper target: ~38.8% F1")
print("  This is the baseline CoT FT must beat.")

del model_lo; torch.cuda.empty_cache()


LabelOnly inference:   0%|          | 0/458 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



PART A RESULT — Qwen Label-Only FT
                       Model Accuracy Precision Recall     F1 ms/sample
Qwen2.5-1.5B (label-only FT)   0.9410    0.8849 0.9179 0.9011    389.61

→ Paper target: ~38.8% F1
  This is the baseline CoT FT must beat.


In [10]:
# ══════════════════════════════════════════════════════════════
# PART B — Qwen CoT FT  (main experiment — reasoning + label)
# Paper target: ~86.0% F1
# ══════════════════════════════════════════════════════════════
print("Loading Qwen2.5-1.5B for CoT fine-tuning...")

model_cot = AutoModelForCausalLM.from_pretrained(
    DECODER_MODEL, quantization_config=BNB_CFG,
    device_map="auto", trust_remote_code=True)

model_cot = get_peft_model(model_cot, lora_cfg)
model_cot.print_trainable_parameters()

COT_BATCH      = 2
COT_GRAD_ACCUM = 4   # effective batch = 8
COT_LR         = 2e-4

tr_dl_cot = DataLoader(
    CoTDataset(df_tr["text"].values, df_tr["rationale"].values, tok),
    batch_size=COT_BATCH, shuffle=True)

opt_cot   = AdamW(filter(lambda p: p.requires_grad, model_cot.parameters()), lr=COT_LR)
sched_cot = get_linear_schedule_with_warmup(
    opt_cot,
    (len(tr_dl_cot)//COT_GRAD_ACCUM)//10,
    (len(tr_dl_cot)//COT_GRAD_ACCUM)*EPOCHS)

for ep in range(1, EPOCHS + 1):
    model_cot.train(); total = 0; opt_cot.zero_grad()
    for step, (be, lbl) in enumerate(tqdm(tr_dl_cot, desc=f"CoT ep{ep}"), 1):
        be  = {k: v.to(DEVICE) for k, v in be.items()}
        out = model_cot(**be, labels=lbl.to(DEVICE))
        (out.loss / COT_GRAD_ACCUM).backward()
        total += out.loss.item()
        if step % COT_GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model_cot.parameters(), 1.0)
            opt_cot.step(); sched_cot.step(); opt_cot.zero_grad()
    print(f"  Epoch {ep} loss: {total/len(tr_dl_cot):.4f}")


Loading Qwen2.5-1.5B for CoT fine-tuning...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


CoT ep1:   0%|          | 0/1830 [00:00<?, ?it/s]

  Epoch 1 loss: 0.4636


CoT ep2:   0%|          | 0/1830 [00:00<?, ?it/s]

  Epoch 2 loss: 0.0032


CoT ep3:   0%|          | 0/1830 [00:00<?, ?it/s]

  Epoch 3 loss: 0.0026


In [11]:
PROMPT_COT = "Analyse this email for phishing indicators:\n{}\n\nAnalysis:"

model_cot.eval(); preds_cot = []; sample_outputs = []
t0 = time.time()

for idx, text in enumerate(tqdm(df_te["text"].values, desc="CoT inference")):
    inputs = tok(PROMPT_COT.format(text[:360]),
                 return_tensors="pt", truncation=True, max_length=420).to(DEVICE)
    with torch.no_grad():
        out = model_cot.generate(
            **inputs, max_new_tokens=120,
            pad_token_id=tok.eos_token_id, do_sample=False)
    decoded = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    preds_cot.append(extract_label(decoded))
    if idx < 5:
        sample_outputs.append({
            "true"      : df_te["label"].values[idx],
            "predicted" : preds_cot[-1],
            "explanation": decoded
        })

ms_cot = (time.time() - t0) / len(df_te) * 1000
RESULT_QWEN_COT = evaluate(df_te["label"].values, preds_cot,
                            "Qwen2.5-1.5B (CoT FT)", ms_cot)

# ── Show sample explanations ──
print("\n── Sample model explanations (first 5 test emails) ──")
for i, s in enumerate(sample_outputs):
    status = "✓" if s["true"] == s["predicted"] else "✗"
    label  = "PHISHING" if s["true"] == 1 else "LEGITIMATE"
    print(f"\n[{i+1}] {status} True={label}  Predicted={'PHISHING' if s['predicted']==1 else 'LEGITIMATE'}")
    print(f"     {s['explanation'][:300]}")

print("\n" + "="*60)
print("PART B RESULT — Qwen CoT FT")
print("="*60)
print(pd.DataFrame([RESULT_QWEN_COT]).to_string(index=False))
print("\n→ Paper target: ~86.0% F1  (up from ~38.8% label-only)")
print(classification_report(df_te["label"].values, preds_cot,
                             target_names=["Legitimate", "Phishing"]))

del model_cot; torch.cuda.empty_cache()


CoT inference:   0%|          | 0/458 [00:00<?, ?it/s]


── Sample model explanations (first 5 test emails) ──

[1] ✓ True=LEGITIMATE  Predicted=LEGITIMATE
      This email contains some phrases that could appear suspicious in isolation. However, the overall context, tone, and structure suggest a legitimate message. No deceptive intent is evident when the full content is considered. Classification: LEGITIMATE

[2] ✓ True=LEGITIMATE  Predicted=LEGITIMATE
      This email contains some phrases that could appear suspicious in isolation. However, the overall context, tone, and structure suggest a legitimate message. No deceptive intent is evident when the full content is considered. Classification: LEGITIMATE

[3] ✓ True=LEGITIMATE  Predicted=LEGITIMATE
      This email contains some phrases that could appear suspicious in isolation. However, the overall context, tone, and structure suggest a legitimate message. No deceptive intent is evident when the full content is considered. Classification: LEGITIMATE

[4] ✓ True=PHISHING  Predicted=PHISHIN

In [13]:
# ══════════════════════════════════════════════════════════════
# FINAL COMPARISON TABLE
# Paste RESULT_BERT from Notebook 1 into the list below
# ══════════════════════════════════════════════════════════════

# Replace this dict with the actual values from Notebook 1
RESULT_BERT = {
    "Model"     : "BERT-base (label-only FT)",
    "Accuracy"  : "paste",
    "Precision" : "paste",
    "Recall"    : "paste",
    "F1"        : "paste",
    "ms/sample" : "paste",
}

summary = pd.DataFrame([RESULT_BERT, RESULT_QWEN_LO, RESULT_QWEN_COT])

print("\n" + "="*70)
print("SCENARIO 4 — FULL COMPARISON TABLE")
print("="*70)
print(summary.to_string(index=False))

print("\n── Paper targets (Yao et al. 2025, arXiv 2505.00034) ──")
print()
print("── Core finding ──")
print("  Label-only → CoT delta        →  +47.2pp F1")
print("  CoT also produces explanations BERT can never give.")



SCENARIO 4 — FULL COMPARISON TABLE
                       Model Accuracy Precision Recall     F1 ms/sample
   BERT-base (label-only FT)    paste     paste  paste  paste     paste
Qwen2.5-1.5B (label-only FT)   0.9410    0.8849 0.9179 0.9011    389.61
       Qwen2.5-1.5B (CoT FT)   0.9563    0.9014 0.9552 0.9275   3966.77

── Paper targets (Yao et al. 2025, arXiv 2505.00034) ──

── Core finding ──
  Label-only → CoT delta        →  +47.2pp F1
  CoT also produces explanations BERT can never give.
